# Lab 5: Batch Gradient Descent for SLR / OLS and SLR Batch GD

## Learning Objectives

By the end of this lab, you should be able to:

- Explain Batch Gradient Descent.
- Implement Batch Gradient Descent for Simple Linear Regression.
- Compute gradients for the slope and intercept.
- Update parameters using a learning rate.
- Track Mean Squared Error during training.
- Compare Batch Gradient Descent with the SLR/OLS closed-form solution.
- Extend Gradient Descent from SLR to Multiple Linear Regression.
- Learn one weight for each feature in MLR.
- Compare SLR / OLS and SLR Batch GD using MAE, MSE, RMSE, and R².

### Lab structure

**SLR → OLS → Batch Gradient Descent → MLR → Batch Gradient Descent for MLR**

The SLR/OLS functions follow the F1 format. Data is converted to NumPy arrays once in the notebook before numerical functions are called.


## Step 1: Import Libraries and Functions

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from slr_ols_student import (
    find_mean,
    find_slope_intercept,
    predict_lr,
    mae,
    mse,
    rmse,
    r_square,
    validate_xy
)

from gradient_descent import (
    predict,
    compute_residual,
    compute_gradient,
    compute_loss,
    update_parameters,
    batch_gradient_descent
)

from mlr_gradient_descent import (
    predict_mlr,
    compute_mlr_residual,
    compute_mlr_gradient,
    compute_mlr_loss,
    update_mlr_parameters,
    batch_gradient_descent_mlr
)

print("Setup complete.")


## Step 2: Download and Load the Dataset

In [ ]:
file_id = "1t5mmVocO1_fGXGqRftpekRom9yxq-ML4"

if not os.path.exists("student_clean_dataset.csv"):
    import gdown
    gdown.download(
        f"https://drive.google.com/uc?id={file_id}",
        "student_clean_dataset.csv",
        quiet=False
    )

df = pd.read_csv("student_clean_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())


## Step 3: SLR Data Preparation

For SLR, use:

- Feature: `Assignments`
- Target: `Exam_Score`


In [ ]:
X = df[["Assignments"]]
y = df["Exam_Score"]

validate_xy(X, y)

# Convert once in the notebook.
x = X["Assignments"].to_numpy(dtype=float)
y_values = y.to_numpy(dtype=float)

print("x type:", type(x))
print("y type:", type(y_values))
print("x shape:", x.shape)
print("y shape:", y_values.shape)


## Step 4: Visualize the SLR Data

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(x, y_values, alpha=0.6)
plt.xlabel("Assignments")
plt.ylabel("Exam Score")
plt.title("Assignments vs Exam Score")
plt.grid(True)
plt.show()


## Step 5: SLR / OLS Closed-Form Solution

The OLS solution provides the reference solution for Gradient Descent.

$$m = \frac{\sum (x_i-\bar{x})(y_i-\bar{y})}{\sum (x_i-\bar{x})^2}$$

$$b = \bar{y} - m\bar{x}$$


In [ ]:
x_mean = find_mean(x)
y_mean = find_mean(y_values)

slr_m, slr_b = find_slope_intercept(
    x, y_values, x_mean, y_mean
)

slr_pred = predict_lr(x, slr_m, slr_b)

print("SLR / OLS slope     :", slr_m)
print("SLR / OLS intercept :", slr_b)


## Step 6: Evaluate SLR / OLS

In [ ]:
slr_mae = mae(y_values, slr_pred)
slr_mse = mse(y_values, slr_pred)
slr_rmse = rmse(y_values, slr_pred)
slr_r2 = r_square(y_values, slr_pred, y_mean)

slr_results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "SLR / OLS": [slr_mae, slr_mse, slr_rmse, slr_r2]
})

display(slr_results)


## Step 7: Batch Gradient Descent for SLR

The functions receive NumPy arrays that were converted once in Step 3.

For MSE:

$$J(m,b)=\frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i-y_i)^2$$

Gradients:

$$\frac{\partial J}{\partial m}=\frac{2}{n}\sum x_i(\hat{y}_i-y_i)$$

$$\frac{\partial J}{\partial b}=\frac{2}{n}\sum(\hat{y}_i-y_i)$$

Parameter updates:

$$m \leftarrow m-\alpha\frac{\partial J}{\partial m}$$

$$b \leftarrow b-\alpha\frac{\partial J}{\partial b}$$


In [ ]:
learning_rate = 0.001
epochs = 30000

initial_m = 0.0
initial_b = 0.0

bgd_m, bgd_b, loss_history = batch_gradient_descent( x, y_values, initial_m, initial_b, learning_rate, epochs)

print("Final Batch GD slope     :", bgd_m)
print("Final Batch GD intercept :", bgd_b)


## Step 8: Evaluate SLR Batch Gradient Descent

In [ ]:
bgd_pred = predict(x, bgd_m, bgd_b)

bgd_mae = mae(y_values, bgd_pred)
bgd_mse = mse(y_values, bgd_pred)
bgd_rmse = rmse(y_values, bgd_pred)
bgd_r2 = r_square(y_values, bgd_pred, y_mean)

bgd_results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "Batch GD": [bgd_mae, bgd_mse, bgd_rmse, bgd_r2]
})

display(bgd_results)


## Step 9: Compare SLR / OLS and Batch Gradient Descent

In [ ]:
slr_gd_comparison = pd.DataFrame({
    "Quantity": ["Slope", "Intercept", "MAE", "MSE", "RMSE", "R2"],
    "SLR / OLS": [slr_m, slr_b, slr_mae, slr_mse, slr_rmse, slr_r2],
    "Batch GD": [bgd_m, bgd_b, bgd_mae, bgd_mse, bgd_rmse, bgd_r2]
})

display(slr_gd_comparison)

print("Absolute slope difference:",
      abs(slr_m - bgd_m))

print("Absolute intercept difference:",
      abs(slr_b - bgd_b))


## Step 10: Plot Batch GD Convergence

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("SLR Batch Gradient Descent Convergence")
plt.grid(True)
plt.show()


## Step 11: Plot SLR / OLS and Batch GD Lines

In [ ]:
sort_idx = np.argsort(x)
x_sorted = x[sort_idx]

slr_line = predict_lr(x_sorted, slr_m, slr_b)
bgd_line = predict(x_sorted, bgd_m, bgd_b)

plt.figure(figsize=(9, 6))
plt.scatter(x, y_values, alpha=0.6, label="Observed Data")
plt.plot(x_sorted, slr_line, linewidth=2, label="SLR / OLS")
plt.plot(x_sorted, bgd_line, linewidth=2, linestyle="--", label="Batch GD")
plt.xlabel("Assignments")
plt.ylabel("Exam Score")
plt.title("SLR / OLS vs Batch Gradient Descent")
plt.legend()
plt.grid(True)
plt.show()


# Core Lab Conclusion

The core experiment compares **SLR / OLS with SLR / Batch Gradient Descent** using the same feature and target.

The MLR Batch Gradient Descent section that follows is an **extra/miscellaneous extension** and is evaluated independently.

# Part B: Extra / Miscellaneous — MLR with Batch Gradient Descent

This section is an extension of the main SLR experiment. It is **not part of the SLR/OLS vs Batch Gradient Descent comparison**.

SLR remains the core experiment. Here we extend the Batch Gradient Descent idea to Multiple Linear Regression (MLR).

SLR uses one feature. MLR uses multiple features:

$$\hat{y}=b+w_1x_1+w_2x_2+...+w_px_p$$

For this extra section, use the four numeric predictors specified in M1:

- `Study_Hours`
- `Attendance`
- `Assignments`
- `Age`

Target: `Exam_Score`.

## Step 12: Select MLR Features

In [ ]:
#TODO
# mlr_features = []

X_mlr = df[mlr_features]
y_mlr = df["Exam_Score"]

validate_xy(X_mlr, y_mlr)

display(X_mlr.head())


## Step 13: Convert MLR Data Once

The conversion is intentionally done here, in the notebook.

All MLR numerical functions below assume `X_mlr_values` and `y_mlr_values` are already NumPy arrays.


In [ ]:
#TODO
# X_mlr_values = 
# y_mlr_values = 

print("X_mlr type:", type(X_mlr_values))
print("y_mlr type:", type(y_mlr_values))
print("X_mlr shape:", X_mlr_values.shape)
print("y_mlr shape:", y_mlr_values.shape)


## Step 14: Initialize MLR Parameters

If there are `p` features, the design matrix `H` contains:

- one column of ones for the intercept
- `p` feature columns

Therefore, the parameter vector contains `p + 1` values.


In [ ]:
#TODO
# n_features = 

initial_weights = np.zeros(n_features)

learning_rate_mlr = 0.00001
epochs_mlr = 10000

print("Number of MLR parameters:", n_features)
print("Initial weights:", initial_weights)


## Step 15: Batch Gradient Descent for MLR

For MLR, the intercept and all feature weights are stored together in one parameter vector.
The design matrix `H` is formed by adding a column of ones to `X`.

$$\nabla J = \frac{2}{n}H^T(\hat{y}-y)$$


In [ ]:
mlr_weights, mlr_loss_history = batch_gradient_descent_mlr(
    X_mlr_values,
    y_mlr_values,
    initial_weights,
    learning_rate_mlr,
    epochs_mlr
)

print("\nFinal MLR parameters:")
for feature, weight in zip(["Intercept"] + mlr_features, mlr_weights):
    print(f"{feature:20s}: {weight}")


## Step 16: MLR Predictions and Evaluation

In [ ]:
mlr_pred = predict_mlr(
    X_mlr_values,
    mlr_weights
)

mlr_y_mean = find_mean(y_mlr_values)

mlr_mae = mae(y_mlr_values, mlr_pred)
mlr_mse = mse(y_mlr_values, mlr_pred)
mlr_rmse = rmse(y_mlr_values, mlr_pred)
mlr_r2 = r_square(y_mlr_values, mlr_pred, mlr_y_mean)

mlr_results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2"],
    "MLR Batch GD": [mlr_mae, mlr_mse, mlr_rmse, mlr_r2]
})

display(mlr_results)


## Step 17: Inspect MLR Coefficients

In [ ]:
mlr_coefficients = pd.DataFrame({
    "Parameter": ["Intercept"] + mlr_features,
    "Coefficient": mlr_weights
})

display(mlr_coefficients)


## Step 18: Plot MLR Convergence

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlr_loss_history)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("MLR Batch Gradient Descent Convergence")
plt.grid(True)
plt.show()


# Final Checklist

- [ ] Loaded the student dataset.
- [ ] Validated the Pandas feature and target data.
- [ ] Converted SLR `X` and `y` to NumPy arrays once in the notebook.
- [ ] Calculated SLR/OLS slope and intercept.
- [ ] Evaluated SLR/OLS.
- [ ] Implemented Batch Gradient Descent for SLR.
- [ ] Compared SLR/OLS with Batch GD.
- [ ] Plotted the SLR Batch GD loss curve.
- [ ] Plotted the two SLR regression lines.
- [ ] Selected the four MLR features from M1.
- [ ] Converted MLR data to NumPy arrays once in the notebook.
- [ ] Implemented Batch Gradient Descent for multiple features.
- [ ] Examined MLR coefficients.
- [ ] Evaluated MLR.
- [ ] Plotted MLR convergence.
- [ ] Compared SLR / OLS and SLR Batch GD.
- [ ] Completed the analysis questions.
